In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import CNKI affiliation name

In [ ]:
cnki = pd.read_csv(dataset_config['path_processed'] + 'CNKI/06_CNKI_DID.csv')
print(f"Number of rows in cnki: {cnki.shape[0]}")

In [ ]:
cnki_aff_stdname = pd.read_csv(dataset_config['path_processed'] + 'CNKI/06_CNKI_DID.csv', usecols=['firm_name_qcc']).drop_duplicates()
cnki_aff_stdname

## Import CSMAR and subs name

In [ ]:
CSMAR1 = pd.read_csv(dataset_config['path_csmar'] + 'STK_NotesSubJoint.csv', sep=',', usecols=['Symbol', 'RalatedParty', 'Relationship'])
CSMAR_subset1 = CSMAR1[CSMAR1["Relationship"] == "上市公司的子公司"].drop_duplicates()

CSMAR2 = pd.read_csv(dataset_config['path_csmar'] + 'STK_NotesSubJoint1.csv', sep=',', usecols=['Symbol', 'RalatedParty', 'Relationship'])
CSMAR_subset2 = CSMAR2[CSMAR2["Relationship"] == "上市公司的子公司"].drop_duplicates()

In [ ]:
CSMAR_sub = pd.concat([CSMAR_subset1, CSMAR_subset2], ignore_index=True)
CSMAR_sub = CSMAR_sub.drop(columns=["Relationship"])
CSMAR_sub

In [ ]:
CSMAR_sub.to_csv(dataset_config['path_processed'] + "WOS/CSMAR_subs_name.csv", index=False) # Export for WOS

## Fuzzy Matching

In [ ]:
"""
# ===============================
# 0. Imports
# ===============================
# !pip install rapidfuzz tqdm

import re
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

tqdm.pandas()  # enable progress bar for pandas apply


# ======================================
# 1. Cleaning function for company names
# ======================================
def clean_name(name: str) -> str:
    """
    Clean Chinese company names:
    - remove common legal suffixes
    - remove parentheses and their content
    - remove English letters, digits, spaces, and simple punctuation
    """
    if pd.isna(name):
        return ""

    name = str(name)

    # Remove common company suffixes
    suffixes = [
        "股份有限公司",
        "有限责任公司",
        "集团有限公司",
        "集团有限责任公司",
        "有限公司",
        "公司",
        "集团",
        "（有限合伙）",
        "(有限合伙)"
    ]
    for s in suffixes:
        name = name.replace(s, "")

    # Remove anything inside Chinese or English parentheses
    name = re.sub(r"（.*?）", "", name)
    name = re.sub(r"\(.*?\)", "", name)

    # Remove English letters and digits
    name = re.sub(r"[A-Za-z0-9]", "", name)

    # Remove spaces and simple punctuation
    name = re.sub(r"[·\.\,\s\-—_]", "", name)

    return name.strip()


# =====================================
# 2. Prepare cleaned columns for both DF
# =====================================
cnki_aff_stdname = cnki_aff_stdname.copy()
cnki_aff_stdname["clean_name"] = cnki_aff_stdname["firm_name_qcc"].astype(str).apply(clean_name)

CSMAR_sub = CSMAR_sub.copy()
CSMAR_sub["clean_RP"] = CSMAR_sub["RalatedParty"].astype(str).apply(clean_name)


# ==================================================
# 3. Prepare choices (clean names) for fuzzy matching
# ==================================================
choices_clean = cnki_aff_stdname["clean_name"].tolist()


# ===========================================
# 4. Fuzzy matching function (one company)
# ===========================================
def fuzzy_match_one(clean_name_target: str, threshold: int = 80) -> pd.Series:
    """
    Fuzzy-match one cleaned company name against cleaned CNKI list.
    Returns:
        - best matched original firm_name_qcc from cnki_aff_stdname (or NA)
        - similarity score
    """
    if pd.isna(clean_name_target) or clean_name_target == "":
        return pd.Series([pd.NA, 0.0])

    match_str, score, idx = process.extractOne(
        clean_name_target,
        choices_clean,
        scorer=fuzz.token_sort_ratio
    )

    if score >= threshold:
        matched_original = cnki_aff_stdname.iloc[idx]["firm_name_qcc"]
        return pd.Series([matched_original, float(score)])
    else:
        return pd.Series([pd.NA, float(score)])


# =====================================
# 5. Run fuzzy matching on FULL sample
# =====================================
CSMAR_sub[["firm_name_qcc_match", "match_score"]] = (
    CSMAR_sub["clean_RP"].progress_apply(fuzzy_match_one)
)

# Optional: see high-quality matches
CSMAR_sub_high = CSMAR_sub[CSMAR_sub["match_score"] >= 95].sort_values("match_score", ascending=False)
CSMAR_sub_high
"""

In [ ]:
# save results
# CSMAR_sub.to_csv(dataset_config['path_processed'] + "CNKI/CSMAR_CNKI_fuzzy_matched_full.csv", index=False)

## Merge with CNKI

In [ ]:
CSMAR_sub = pd.read_csv(dataset_config['path_processed'] + 'CNKI/CSMAR_CNKI_fuzzy_matched_full.csv')
CSMAR_sub

In [ ]:
CSMAR_sub_high = CSMAR_sub[CSMAR_sub["match_score"] >= 95].sort_values("match_score", ascending=False)
CSMAR_sub_high

In [ ]:
CSMAR_merge = CSMAR_sub_high[['firm_name_qcc_match', 'Symbol']]
CSMAR_merge = CSMAR_merge.rename(columns={"firm_name_qcc_match": "firm_name_qcc"}).drop_duplicates(subset="firm_name_qcc", keep="first")
CSMAR_merge

In [ ]:
cnki_match_csmar = pd.merge(cnki, CSMAR_merge, on='firm_name_qcc', how='left')
print(f"Number of rows in cnki: {cnki_match_csmar.shape[0]}")

In [ ]:
cnki_match_csmar

In [ ]:
# 1. Keep non-missing symbol rows
cnki_csmar = cnki_match_csmar[cnki_match_csmar["Symbol"].notna()].copy()

# 2. Group and sum pub_num by Symbol × year
df_sum = (
    cnki_csmar.groupby(["Symbol", "year"], as_index=False)
    .agg(
        pub_num_sum=('pub_num', 'sum'),  # Sum of pub_num
        fractional_num_sum=('pub_num_fractional', 'sum')  # Sum of pub_num_fractional
    )
    .sort_values(["Symbol", "year"])
)

df_sum

In [ ]:
df_sum.to_csv(dataset_config['path_processed'] + "CNKI/CNKI_public_withsubs_year.csv", index=False)